In [ ]:
import pandas as pd
import numpy as np
import pickle

interactions = pd.read_parquet("data/processed/interactions.parquet")
tracks_df = pd.read_parquet("data/processed/tracks.parquet")

In [6]:

import pandas as pd
from pathlib import Path
import sys
import importlib

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if "interactions" not in globals():
    interactions = pd.read_parquet("../data/processed/interactions.parquet")
if "tracks_df" not in globals():
    tracks_df = pd.read_parquet("../data/processed/tracks.parquet")

import src.data.preprocesssor as preprocesssor
preprocesssor = importlib.reload(preprocesssor)
build_interaction_matrix = preprocesssor.build_interaction_matrix

matrix, user_enc, artist_enc = build_interaction_matrix(interactions)

print(f"Matrix shape:    {matrix.shape}")
print(f"Stored values:   {matrix.nnz:,}")
print(f"Memory (MB):     {matrix.data.nbytes / 1e6:.1f}")
print(f"Dtype:           {matrix.dtype}")

Matrix shape:    (74999, 145778)
Stored values:   3,664,980
Memory (MB):     14.7
Dtype:           float32


In [8]:
import numpy as np
print(f"Value range: [{matrix.data.min():.4f}, {matrix.data.max():.4f}]")


sample_rows = np.random.choice(matrix.shape[0], size=5, replace=False)
for row in sample_rows:
    row_data = matrix.getrow(row)
    norm = np.sqrt(row_data.multiply(row_data).sum())
    print(f"Row {row} norm: {norm:.6f}")

Value range: [0.0120, 1.0000]
Row 28727 norm: 1.000000
Row 3219 norm: 1.000000
Row 13010 norm: 1.000000
Row 60005 norm: 1.000000
Row 29255 norm: 1.000000


In [9]:
from src.data.preprocesssor import build_popularity_scores

popularity_df = build_popularity_scores(interactions)
print(popularity_df.head(10))
print(f"\nPopularity score range: [{popularity_df['popularity_score'].min():.4f}, {popularity_df['popularity_score'].max():.4f}]")

             artist_name  total_plays  unique_listeners  \
0              radiohead      5826626             16149   
1            the beatles      6335329             15897   
2               coldplay      3552759             14006   
3  red hot chili peppers      2921496             10301   
4                   muse      3219982              9802   
5              metallica      3160071              9597   
6             pink floyd      3269778              9309   
7            the killers      2022627              8611   
8                nirvana      2074015              8439   
9            linkin park      2589909              8289   

   avg_plays_per_listener  popularity_score  
0              360.804136          1.000000  
1              398.523558          0.999993  
2              253.659789          0.999986  
3              283.612853          0.999979  
4              328.502550          0.999973  
5              329.276962          0.999966  
6              351.249114   

In [10]:
from src.data.features import build_content_features

content_df, scaler = build_content_features(tracks_df, popularity_df)

matched = content_df["has_content_features"].sum()
total = len(content_df)
print(f"Artists with content features: {matched:,} / {total:,} ({matched/total*100:.1f}%)")
print(f"\nSample matched rows:")
print(content_df[content_df["has_content_features"]].head(3))
print(f"\nSample unmatched rows:")
print(content_df[~content_df["has_content_features"]].head(3))

Artists with content features: 7,505 / 145,778 (5.1%)

Sample matched rows:
   artist_name  total_plays  unique_listeners  avg_plays_per_listener  \
0    radiohead      5826626             16149              360.804136   
1  the beatles      6335329             15897              398.523558   
2     coldplay      3552759             14006              253.659789   

   popularity_score  artist_norm  danceability    energy  loudness  \
0          1.000000    radiohead     -1.021637 -1.011130 -0.412334   
1          0.999993  the beatles     -0.187181 -0.559672 -0.210364   
2          0.999986     coldplay     -0.497510 -0.500392  0.187055   

   speechiness  acousticness  instrumentalness  liveness   valence     tempo  \
0    -0.511230     -0.403161         -0.183448 -0.590341 -1.193322 -1.047380   
1    -0.388207      0.193464         -0.509950 -0.032264  0.540465 -0.034255   
2    -0.540847     -0.281078         -0.589892 -0.398645 -0.980313  0.509785   

   has_content_features  
0  

In [12]:
from scipy.sparse import save_npz
import pickle

save_npz("../data/processed/interaction_matrix.npz", matrix)


with open("../data/processed/user_encoder.pkl", "wb") as f:
    pickle.dump(user_enc, f)

with open("../data/processed/artist_encoder.pkl", "wb") as f:
    pickle.dump(artist_enc, f)

popularity_df.to_parquet("../data/processed/artist_popularity.parquet", index=False)
content_df.to_parquet("../data/processed/content_features.parquet", index=False)

# Scaler — save for Phase 5 inference
with open("../data/processed/audio_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("All artifacts saved.")

All artifacts saved.
